# Logistic Regression: Heart Failure Prediction

## Step 1: Import Libraries

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

## Step 2: Load and Explore Data

In [ ]:
df = pd.read_csv(r"C:\Users\24108218\Documents\Downloads\h\heart.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Step 3: Data Cleaning
In this dataset, 0 in `Cholesterol` and `RestingBP` means the value is missing.

In [ ]:
print(df.isnull().sum())
df = df.drop_duplicates()

df = df[df['RestingBP'] > 0]                                   # remove invalid BP row
chol_median = df.loc[df['Cholesterol'] > 0, 'Cholesterol'].median()
df['Cholesterol'] = df['Cholesterol'].replace(0, chol_median)  # fill missing cholesterol
print(df.shape)

## Step 4: Target Distribution and Heatmap

In [ ]:
print(df['HeartDisease'].value_counts())
sns.countplot(x='HeartDisease', data=df)
plt.title('Heart Disease (0 = No, 1 = Yes)')
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## Step 4b: Define Feature Columns

In [ ]:
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
cat_cols = ['Sex', 'ChestPainType', 'ExerciseAngina', 'ST_Slope']

print(len(num_cols) + len(cat_cols), "features")

## Step 5: Split, Preprocess and Train

In [ ]:
X = df[num_cols + cat_cols]
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

model = Pipeline([
    ('prep', preprocess),
    ('lr', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

## Step 6: Model Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Healthy','Heart Disease']))

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['Healthy','Heart Disease']).plot(cmap='Reds')
plt.title('Confusion Matrix')
plt.show()

## Step 7: Test One Prediction and Save the Model

In [ ]:
sample = pd.DataFrame([{
    'Age': 58, 'RestingBP': 140, 'Cholesterol': 240, 'FastingBS': 1,
    'MaxHR': 120, 'Oldpeak': 2.0, 'Sex': 'M', 'ChestPainType': 'ASY',
    'ExerciseAngina': 'Y', 'ST_Slope': 'Flat'}])
print("Prediction:", "Heart Disease" if model.predict(sample)[0] == 1 else "Healthy")
print("Probability:", model.predict_proba(sample)[0][1])

joblib.dump(model, 'heart_model.pkl')
print("Model saved: heart_model.pkl")